In [0]:
CATALOG = "nestle_sales_databricks"

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

LAYERS = ["bronze", "silver", "gold"]

print("Catalog:", CATALOG)
print("Storage:", BASE_PATH)

Catalog: nestle_sales_databricks
Storage: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



In [0]:
import re
from delta.tables import DeltaTable
registration_results = []
for layer in LAYERS:
    layer_path = f"{BASE_PATH}{layer}/"
    schema_name = f"{CATALOG}.{layer}"
    try:
        folders = dbutils.fs.ls(layer_path)
    except Exception as e:
        print(f"Cannot access {layer_path}: {e}")
        continue
    for folder in folders:
        if not folder.isDir():
            continue
        table_name = folder.name.rstrip("/")
        # Use simple folder names as SQL identifiers.
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table_name):
            print(f"Skipped unsupported table name: {table_name}")
            continue
        table_path = folder.path
        try:
            if not DeltaTable.isDeltaTable(spark, table_path):
                print(f"Skipped non-Delta folder: {table_path}")
                continue
            full_table_name = f"{schema_name}.{table_name}"
            # Register only if the catalog table doesn't already exist.
            exists = spark.catalog.tableExists(full_table_name)
            if not exists:
                escaped_path = table_path.replace("'", "''")
                spark.sql(f"""
                    CREATE TABLE {full_table_name}
                    USING DELTA
                    LOCATION '{escaped_path}'
                """)
                status = "REGISTERED"
            else:
                status = "ALREADY EXISTS — NOT MODIFIED"
            registration_results.append(
                (layer, table_name, table_path, status)
            )
            print(f"{full_table_name}: {status}")
        except Exception as e:
            print(f"Failed to register {table_name}: {e}")
print("Setup scan completed.")

nestle_sales_databricks.bronze.customers: REGISTERED
nestle_sales_databricks.bronze.orders: ALREADY EXISTS — NOT MODIFIED
nestle_sales_databricks.bronze.products: REGISTERED
nestle_sales_databricks.silver.customers: REGISTERED
nestle_sales_databricks.silver.orders: REGISTERED
nestle_sales_databricks.silver.products: REGISTERED
nestle_sales_databricks.gold.customer_sales: REGISTERED
nestle_sales_databricks.gold.daily_sales: REGISTERED
nestle_sales_databricks.gold.market_sales: REGISTERED
nestle_sales_databricks.gold.product_sales: REGISTERED
Setup scan completed.


In [0]:
for layer in LAYERS:
    print(f"\n===== {layer.upper()} TABLES =====")

    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{layer}"
    ).show(truncate=False)


===== BRONZE TABLES =====
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|bronze  |customers|false      |
|bronze  |orders   |false      |
|bronze  |products |false      |
+--------+---------+-----------+


===== SILVER TABLES =====
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|silver  |customers|false      |
|silver  |orders   |false      |
|silver  |products |false      |
+--------+---------+-----------+


===== GOLD TABLES =====
+--------+--------------+-----------+
|database|tableName     |isTemporary|
+--------+--------------+-----------+
|gold    |customer_sales|false      |
|gold    |daily_sales   |false      |
|gold    |market_sales  |false      |
|gold    |product_sales |false      |
+--------+--------------+-----------+



In [0]:
CATALOG = "nestle_sales_databricks"

tables_by_layer = {
    "bronze": ["customers", "orders", "products"],
    "silver": ["customers", "orders", "products"],
    "gold": [
        "customer_sales",
        "daily_sales",
        "market_sales",
        "product_sales"
    ]
}

verification_results = []

for layer, tables in tables_by_layer.items():
    for table in tables:
        full_name = f"{CATALOG}.{layer}.{table}"

        try:
            detail = (
                spark.sql(f"DESCRIBE DETAIL {full_name}")
                .first()
                .asDict()
            )

            row_count = spark.table(full_name).count()

            verification_results.append((
                layer,
                table,
                detail.get("format"),
                detail.get("location"),
                row_count,
                "OK"
            ))

        except Exception as e:
            verification_results.append((
                layer, table, None, None, None, str(e)[:200]
            ))

result_df = spark.createDataFrame(
    verification_results,
    [
        "layer", "table", "format",
        "location", "row_count", "status"
    ]
)

display(result_df)

layer,table,format,location,row_count,status
bronze,customers,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers,175003,OK
bronze,orders,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders,150003,OK
bronze,products,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products,150003,OK
silver,customers,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/customers,173128,OK
silver,orders,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/orders,65003,OK
silver,products,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/silver/products,120003,OK
gold,customer_sales,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/gold/customer_sales,86008,OK
gold,daily_sales,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/gold/daily_sales,6,OK
gold,market_sales,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/gold/market_sales,9,OK
gold,product_sales,delta,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/gold/product_sales,86007,OK


In [0]:
spark.sql("DESCRIBE DETAIL nestle_sales_databricks.bronze.customers").show(truncate=False)

+------+------------------------------------+----------------------------------------+-----------+-----------------------------------------------------------------------------------------+-----------------------+-------------------+----------------+-----------------+--------+-----------+-------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                    |description|location                                                                                 |createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                           |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------